# Bengaluru House Price Prediction
### Neural Network Model — Real Dataset Version

In [7]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

print('All libraries imported!')

All libraries imported!


## Step 1 — Load Data

In [8]:
df = pd.read_csv('bengaluru_house_data.csv')
print('Shape:', df.shape)

Shape: (13320, 9)


In [9]:
print('\nColumns:', df.columns.tolist())
df.head()


Columns: ['area_type', 'availability', 'location', 'size', 'society', 'total_sqft', 'bath', 'balcony', 'price']


,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


## Step 2 — Understanding the Data

In [10]:
print('Null values:\n', df.isnull().sum())

Null values:
 area_type          0
availability       0
location           1
size              16
society         5502
total_sqft         0
bath              73
balcony          609
price              0
dtype: int64


In [11]:
print('\nPrice stats:')
print(df['price'].describe())


Price stats:
count    13320.000000
mean       112.565627
std        148.971674
min          8.000000
25%         50.000000
50%         72.000000
75%        120.000000
max       3600.000000
Name: price, dtype: float64


In [12]:
print('\nUnique sizes (sample):', df['size'].unique()[:10])


Unique sizes (sample): ['2 BHK' '4 Bedroom' '3 BHK' '4 BHK' '6 Bedroom' '3 Bedroom' '1 BHK'
 '1 RK' '1 Bedroom' '8 Bedroom']


In [13]:
print('\nArea types:', df['area_type'].unique())


Area types: ['Super built-up  Area' 'Plot  Area' 'Built-up  Area' 'Carpet  Area']


## Step 3 — Clean the Data 
* here i took the help of Ai

In [14]:
df['bhk'] = df['size'].str.extract(r'(\d+)').astype(float)
def convert_sqft(val):
    try:
        if '-' in str(val):
            parts = str(val).split('-')
            return (float(parts[0]) + float(parts[1])) / 2
        return float(val)
    except:
        return np.nan

df['total_sqft'] = df['total_sqft'].apply(convert_sqft)
df = df.drop(['size', 'society'], axis=1)
print('Cleaned shape:', df.shape)
df.head()

Cleaned shape: (13320, 8)


,area_type,availability,location,total_sqft,bath,balcony,price,bhk
0,Super built-up Area,19-Dec,Electronic City Phase II,1056.0,2.0,1.0,39.07,2.0
1,Plot Area,Ready To Move,Chikka Tirupathi,2600.0,5.0,3.0,120.00,4.0
2,Built-up Area,Ready To Move,Uttarahalli,1440.0,2.0,3.0,62.00,3.0
3,Super built-up Area,Ready To Move,Lingadheeranahalli,1521.0,3.0,1.0,95.00,3.0
4,Super built-up Area,Ready To Move,Kothanur,1200.0,2.0,1.0,51.00,2.0


## Step 4 — Handling Missing Values

In [15]:
print('Nulls before:\n',df.isnull().sum())

Nulls before:
 area_type         0
availability      0
location          1
total_sqft       46
bath             73
balcony         609
price             0
bhk              16
dtype: int64


In [16]:
df['bath']=df['bath'].fillna(df['bath'].median())

In [17]:
df['balcony']=df['balcony'].fillna(df['balcony'].median())

In [18]:
df['total_sqft']=df['total_sqft'].fillna(df['total_sqft'].median())

In [19]:
df['bhk']=df['bhk'].fillna(df['bhk'].median())

In [20]:
df=df.dropna(subset=['price','location'])

In [21]:
# Remove the top 5% most expensive houses
df = df[df['price'] <= df['price'].quantile(0.95)]
print('Shape after removing expensive outliers:', df.shape)

Shape after removing expensive outliers: (12654, 8)


In [22]:
print('Nulls after:\n', df.isnull().sum())
print('Shape:', df.shape)

Nulls after:
 area_type       0
availability    0
location        0
total_sqft      0
bath            0
balcony         0
price           0
bhk             0
dtype: int64
Shape: (12654, 8)


## Step 6 — Encodeing Categorical Columns
* `location` has hundreds of values — we keep top 20 and group the rest as `'Other'`.

In [23]:
top_locations=df['location'].value_counts().head(40).index
df['location']=df['location'].apply(lambda x: x if x in top_locations else 'Other')
df = pd.get_dummies(df, columns=['location', 'area_type', 'availability'], dtype=int)
print('Shape after encoding:', df.shape)
df.head()

Shape after encoding: (12654, 131)


,total_sqft,bath,balcony,price,bhk,location_7th Phase JP Nagar,location_Akshaya Nagar,location_Banashankari,location_Bannerghatta Road,location_Begur Road,...,availability_21-Oct,availability_21-Sep,availability_22-Dec,availability_22-Jan,availability_22-Jun,availability_22-Mar,availability_22-May,availability_22-Nov,availability_Immediate Possession,availability_Ready To Move
0,1056.0,2.0,1.0,39.07,2.0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2600.0,5.0,3.0,120.00,4.0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,1440.0,2.0,3.0,62.00,3.0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,1521.0,3.0,1.0,95.00,3.0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,1200.0,2.0,1.0,51.00,2.0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


## Step 7 — Check Correlations (lesson learned!)

In [24]:
x = df.drop('price', axis=1)
y = df['price']

## Step 8 — Train/Test Split & Scale

In [25]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [26]:
scale   = StandardScaler()

x_train = scale.fit_transform(x_train)
x_test  = scale.transform(x_test)

print('Train size:', x_train.shape)
print('Test size :', x_test.shape)

Train size: (10123, 130)
Test size : (2531, 130)


## Step 9 — Convert to Tensors
> this part is much confuesd and i took the help of AI 

In [27]:
x_train_t=torch.from_numpy(x_train).float()
y_train_t=torch.from_numpy(y_train.values).float().view(-1, 1)

x_test_t=torch.from_numpy(x_test).float()
y_test_t=torch.from_numpy(y_test.values).float().view(-1, 1)

train_dataset=TensorDataset(x_train_t, y_train_t)
test_dataset=TensorDataset(x_test_t,  y_test_t)

train_load=DataLoader(train_dataset, batch_size=32, shuffle=True)
test_load=DataLoader(test_dataset,  batch_size=32)
print('Input features:', x_train_t.shape[1])

Input features: 130


## Step 10 — Build Model

In [28]:
input_size = x_train_t.shape[1]

model = nn.Sequential(
    nn.Linear(input_size, 128),   
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 64),         
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.BatchNorm1d(32),
    nn.ReLU(),
    nn.Linear(32, 1)
)
print(model)

Sequential(
  (0): Linear(in_features=130, out_features=128, bias=True)
  (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU()
  (3): Dropout(p=0.2, inplace=False)
  (4): Linear(in_features=128, out_features=64, bias=True)
  (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (6): ReLU()
  (7): Linear(in_features=64, out_features=32, bias=True)
  (8): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (9): ReLU()
  (10): Linear(in_features=32, out_features=1, bias=True)
)


In [29]:
loss_fn   = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-4)

## Step 11 — Train with Early Stopping


**Dropout(0.2):**
It randomly switches off 20% of neurons during training. 
This forces the model to not rely too much on any one neuron — which reduces overfitting.

**BatchNorm:**
Normalizes the output of each layer within a batch before passing to the next layer. 
Keeps values in a stable range so training is faster and more stable.

In [ ]:
best_val_loss    = float('inf')
patience         = 30
counter          = 0
# best_model_state = None

for epoch in range(500):
    model.train()
    train_loss = 0
    for x_batch, y_batch in train_load:
        logits = model(x_batch)
        loss   = loss_fn(logits, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss+=loss.item()*x_batch.size(0)
    train_loss /= len(train_load.dataset)
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for x_batch, y_batch in test_load:
            logits=model(x_batch)
            loss=loss_fn(logits, y_batch)
            val_loss+=loss.item()*x_batch.size(0)
    val_loss /= len(test_load.dataset)
    if val_loss < best_val_loss:
        best_val_loss=val_loss
        counter=0
    else:
        counter += 1
    if counter >= patience:
        print(f'Early stopping at epoch {epoch + 1}')
        break
    if (epoch + 1) % 50 == 0:
        print(f'Epoch {epoch+1:>3} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}')
print('\nTraining complete!')

Epoch  50 | Train Loss: 1070.5237 | Val Loss: 1101.5736


## Step 12 — Evaluate

In [ ]:
model.eval()
actual=[] 
pred=[]
with torch.no_grad():
    for xb, yb in test_load:
        output=model(xb)
        actual.extend(yb.squeeze().tolist())
        pred.extend(output.squeeze().tolist())

In [ ]:
actual=np.array(actual)
pred=np.array(pred)

RMSE=np.sqrt(mean_squared_error(actual, pred))
MSE=mean_squared_error(actual, pred)
r2=r2_score(actual, pred)

print('=' * 40)
print(f'  RMSE : {RMSE:.2f} Lakhs')
print(f'  MSE  : {MSE:.2f} Lakhs')
print(f'  R²   : {r2*100:.2f}%')
print('=' * 40)

## Step 13 — Save Model for Streamlit
> this is the work where AI helped me more

In [66]:
import pickle, json

torch.save(model.state_dict(), 'housing_model.pth')

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scale, f)

columns = list(x.columns)
with open('model_columns.json', 'w') as f:
    json.dump(columns, f)

# Save top locations list for Streamlit dropdowns
with open('top_locations.json', 'w') as f:
    json.dump(list(top_locations), f)

print('All files saved!')
print('Files: housing_model.pth, scaler.pkl, model_columns.json, top_locations.json')

All files saved!
Files: housing_model.pth, scaler.pkl, model_columns.json, top_locations.json


## Step 14 — Prediction Function + Streamlit App
> this also whare AI took this weight

In [67]:
def predict_price(location, area_type, availability, total_sqft, bath, balcony, bhk):
    # Step 1: Build input dict
    input_dict = {
        'total_sqft': total_sqft,
        'bath'      : bath,
        'balcony'   : balcony,
        'bhk'       : bhk
    }
    input_df = pd.DataFrame([input_dict])

    # Step 2: One-hot encode — match training columns
    loc_col  = f'location_{location}' if location in list(top_locations) else 'location_Other'
    area_col = f'area_type_{area_type}'
    avail_col = f'availability_{availability}'

    for col in columns:
        if col not in input_df.columns:
            input_df[col] = 0

    if loc_col in columns:
        input_df[loc_col]=1
    if area_col in columns:
        input_df[area_col]=1
    if avail_col in columns:
        input_df[avail_col]=1

    # Step 3: Reorder, scale, predict
    input_df     = input_df[columns]
    input_scaled = scale.transform(input_df)
    tensor       = torch.from_numpy(input_scaled).float()

    model.eval()
    with torch.no_grad():
        result = model(tensor).item()

    return round(result, 2)


# Test prediction
print('Test prediction:')
price = predict_price(
    location     = 'Whitefield',
    area_type    = 'Super built-up Area',
    availability = 'Ready To Move',
    total_sqft   = 1200,
    bath         = 2,
    balcony      = 1,
    bhk          = 2
)
print(f'Predicted Price: ₹{price} Lakhs')

Test prediction:
Predicted Price: ₹71.23 Lakhs


C:\Users\Vishnu\AppData\Local\Temp\ipykernel_12488\797877124.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  input_df[col] = 0
C:\Users\Vishnu\AppData\Local\Temp\ipykernel_12488\797877124.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  input_df[col] = 0
C:\Users\Vishnu\AppData\Local\Temp\ipykernel_12488\797877124.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(ax

In [2]:
app_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import pickle
import json

# Load saved files
with open("scaler.pkl", "rb") as f:
    scale = pickle.load(f)
with open("model_columns.json", "r") as f:
    columns = json.load(f)
with open("top_locations.json", "r") as f:
    top_locations = json.load(f)

# Rebuild model
input_size = len(columns)
model = nn.Sequential(
    nn.Linear(input_size, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.BatchNorm1d(32),
    nn.ReLU(),
    nn.Linear(32, 1)
)

model.load_state_dict(torch.load("housing_model.pth", map_location="cpu"))
model.eval()

def predict_price(location, area_type, availability, total_sqft, bath, balcony, bhk):
    input_df = pd.DataFrame([{"total_sqft": total_sqft, "bath": bath, "balcony": balcony, "bhk": bhk}])
    loc_col   = f"location_{location}" if f"location_{location}" in columns else "location_Other"
    area_col  = f"area_type_{area_type}"
    avail_col = f"availability_{availability}"
    for col in columns:
        if col not in input_df.columns:
            input_df[col] = 0
    if loc_col in columns:   input_df[loc_col]   = 1
    if area_col in columns:  input_df[area_col]  = 1
    if avail_col in columns: input_df[avail_col] = 1
    input_df     = input_df[columns]
    input_scaled = scale.transform(input_df)
    tensor       = torch.from_numpy(input_scaled).float()
    with torch.no_grad():
        result = model(tensor).item()
    return round(result, 2)

# Streamlit UI
st.title("House Price Predictor")
st.markdown("Fill in the details to get a price prediction.")

col1, col2 = st.columns(2)

with col1:
    location     = st.selectbox("Location", sorted(top_locations) + ["Other"])
    area_type    = st.selectbox("Area Type", ["Super built-up Area", "Built-up Area", "Plot Area", "Carpet Area"])
    availability = st.selectbox("Availability", ["Ready To Move", "Immediate Possession", "6 months", "1 Year"])
    bhk          = st.slider("BHK", 0, 10, 0)

with col2:
    total_sqft = st.number_input("Total SqFt", min_value=200, max_value=100000, value=1200)
    bath       = st.slider("Bathrooms", 0, 10, 0)
    balcony    = st.slider("Balconies", 0, 10, 0)

if st.button("💰 Predict Price"):
    price = predict_price(location, area_type, availability, total_sqft, bath, balcony, bhk)
    st.success(f"Predicted Price: ₹ {price} Lakhs")
    st.info(f"Approx: ₹ {round(price/total_sqft*100000):,} per sq.ft")
'''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

print('app.py created!')
print('Run in terminal:  streamlit run app.py')

app.py created!
Run in terminal:  streamlit run app.py
